# Controlled small-data and FP32 experiments

Historical research source, original zero-based cells **38?46**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 38

Position 39 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 38 - SMALL DATASET FOR CONTROLLED ASR TRAINING
#
# 8 training recordings from 8 speakers
# 4 held-out recordings from 4 DIFFERENT speakers
#
# No model training and no checkpoint modifications.
# ============================================================

import json
import random
from pathlib import Path
from IPython.display import Audio, display

SMALL_SET_DIR = DATA_DIR / "head_only_multirecording_v1"
SMALL_SET_DIR.mkdir(parents=True, exist_ok=True)

SMALL_SET_PATH = SMALL_SET_DIR / "selection.json"

TRAIN_COUNT = 8
VAL_COUNT = 4
SELECTION_SEED = 20261019

# XLS-R 300M convolution configuration.
CONV_KERNELS = (10, 3, 3, 3, 3, 2, 2)
CONV_STRIDES = (5, 2, 2, 2, 2, 2, 2)

def count_output_frames(num_samples):
    length = int(num_samples)
    for kernel, stride in zip(CONV_KERNELS, CONV_STRIDES):
        length = (length - kernel) // stride + 1
    return length

def inspect_record(record):

    duration = float(record["duration_seconds"])
    speaker = record.get("speaker_id")

    if not speaker or not (4.0 <= duration <= 18.0):
        return None

    audio_path = Path(AUDIO_DIR) / record["audio_file"]

    if not audio_path.is_file():
        return None

    labels = tokenizer.encode(
        record["transcription"],
        add_special_tokens=False
    )

    if not (25 <= len(labels) <= 140):
        return None

    if tokenizer.unk_token_id in labels:
        return None

    repeated = sum(
        a == b
        for a, b in zip(labels[:-1], labels[1:])
    )

    required = len(labels) + repeated
    available = count_output_frames(
        round(duration * 16000)
    )

    # Leave a conservative alignment margin.
    if available - required < 20:
        return None

    return {
        "record": record,
        "tokens": len(labels),
        "margin": available - required
    }

# ------------------------------------------------------------
# 1. Select recordings or reuse saved selection
# ------------------------------------------------------------

record_lookup = {
    r["id"]: r for r in train_records
}

if SMALL_SET_PATH.exists():

    selection = json.loads(
        SMALL_SET_PATH.read_text(encoding="utf-8")
    )

    train_ids = selection["train_ids"]
    val_ids = selection["val_ids"]

    print("Loaded existing selection.")

else:

    eligible = []

    for record in train_records:
        result = inspect_record(record)
        if result is not None:
            eligible.append(result)

    rng = random.Random(SELECTION_SEED)
    rng.shuffle(eligible)

    selected = []
    used_speakers = set()

    for candidate in eligible:

        record = candidate["record"]
        speaker = record["speaker_id"]

        if speaker in used_speakers:
            continue

        selected.append(record["id"])
        used_speakers.add(speaker)

        if len(selected) == TRAIN_COUNT + VAL_COUNT:
            break

    print("Eligible recordings:", len(eligible))
    print("Selected speakers:", len(selected))

    assert len(selected) == TRAIN_COUNT + VAL_COUNT, (
        "Not enough suitable recordings with distinct speakers."
    )

    train_ids = selected[:TRAIN_COUNT]
    val_ids = selected[TRAIN_COUNT:]

    selection = {
        "seed": SELECTION_SEED,
        "train_ids": train_ids,
        "val_ids": val_ids
    }

    SMALL_SET_PATH.write_text(
        json.dumps(
            selection, ensure_ascii=False, indent=2
        ),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 2. Validate our selected recordings
# ------------------------------------------------------------

assert len(train_ids) == TRAIN_COUNT
assert len(val_ids) == VAL_COUNT

assert len(set(train_ids + val_ids)) == (
    TRAIN_COUNT + VAL_COUNT
)

small_train_records = [
    record_lookup[id_] for id_ in train_ids
]

small_val_records = [
    record_lookup[id_] for id_ in val_ids
]

train_speakers = {
    r["speaker_id"] for r in small_train_records
}
val_speakers = {
    r["speaker_id"] for r in small_val_records
}

assert not (train_speakers & val_speakers)
assert len(train_speakers) == TRAIN_COUNT
assert len(val_speakers) == VAL_COUNT

for record in small_train_records + small_val_records:
    assert inspect_record(record) is not None, (
        f"Missing audio or invalid recording: {record['id']}"
    )

print("\n" + "=" * 60)
print("SMALL DATASET VERIFICATION")
print("=" * 60)

print("Training recordings:", len(small_train_records))
print("Held-out recordings:", len(small_val_records))
print("Overlapping speakers:", len(train_speakers & val_speakers))

print("\nTRAINING RECORDINGS")
for i, r in enumerate(small_train_records, 1):
    print(
        f"{i}. {r['id']} | "
        f"{r['duration_seconds']:.2f}s | "
        f"speaker={r['speaker_id']}"
    )
    print("   ", r["transcription"][:130])

print("\nHELD-OUT RECORDINGS")
for i, r in enumerate(small_val_records, 1):
    print(
        f"{i}. {r['id']} | "
        f"{r['duration_seconds']:.2f}s | "
        f"speaker={r['speaker_id']}"
    )
    print("   ", r["transcription"][:130])

# ------------------------------------------------------------
# 3. Listen to three recordings
# ------------------------------------------------------------

examples_to_listen = [
    ("TRAIN 1", small_train_records[0]),
    ("TRAIN 2", small_train_records[1]),
    ("HELD-OUT 1", small_val_records[0])
]

print("\n" + "=" * 60)
print("AUDIO AND TRANSCRIPT QUALITY CHECK")
print("=" * 60)

for label, record in examples_to_listen:

    print("\n", label)
    print("Recording ID:", record["id"])
    print("Transcription:", record["transcription"])

    audio_file = Path(AUDIO_DIR) / record["audio_file"]

    display(Audio(filename=str(audio_file)))

print("\nSelection saved:", SMALL_SET_PATH)
print("No model weights or checkpoints were modified.")


## Original source cell 39

Position 40 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 39 - MULTI-RECORDING HEAD-ONLY ASR EXPERIMENT
#
# 8 training recordings / 4 held-out recordings
# Original XLS-R encoder is FROZEN.
# Only the CTC prediction head is trained.
#
# Saves checkpoints; does NOT change previous models.
# ============================================================

import json
import gc
import copy
import hashlib
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MAX_UPDATES = 4000
REPORT_EVERY = 250
LEARNING_RATE = 1e-3
BLANK_ID = tokenizer.pad_token_id

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EXPERIMENT_DIR = (
    DATA_DIR / "head_only_multirecording_v1"
)
SELECTION_PATH = EXPERIMENT_DIR / "selection.json"
CHECKPOINT_PATH = EXPERIMENT_DIR / "training_latest.pt"
HISTORY_PATH = EXPERIMENT_DIR / "training_history.json"

assert SELECTION_PATH.exists()
assert Path(BASELINE_DIR).exists()

selection = json.loads(
    SELECTION_PATH.read_text(encoding="utf-8")
)

train_ids = selection["train_ids"]
val_ids = selection["val_ids"]

assert len(train_ids) == 8
assert len(val_ids) == 4
assert len(set(train_ids + val_ids)) == 12

record_by_id = {
    record["id"]: record
    for record in train_records
}

index_by_id = {
    record["id"]: i
    for i, record in enumerate(train_dataset.records)
}

train_speakers = {
    record_by_id[rid]["speaker_id"]
    for rid in train_ids
}
val_speakers = {
    record_by_id[rid]["speaker_id"]
    for rid in val_ids
}

assert not (train_speakers & val_speakers)

selection_signature = hashlib.sha256(
    json.dumps(
        {
            "train": train_ids,
            "val": val_ids,
            "references": {
                rid: record_by_id[rid]["transcription"]
                for rid in train_ids + val_ids
            }
        },
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

print("MULTI-RECORDING EXPERIMENT")
print("=" * 55)
print("Training examples:", len(train_ids))
print("Held-out examples:", len(val_ids))
print("Shared speakers:", 0)
print("Device:", device)

# ------------------------------------------------------------
# 2. Load original pretrained encoder
# ------------------------------------------------------------

print("\nLoading original XLS-R checkpoint...")

base_model = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(device)

base_model.eval()

# Head is copied OUTSIDE inference mode.
head = copy.deepcopy(
    base_model.lm_head
).to(device).float()

assert head.weight.requires_grad
assert not head.weight.is_inference()

# ------------------------------------------------------------
# 3. Cache encoder features for all 12 examples
# ------------------------------------------------------------

def cache_example(record_id):

    example = train_dataset[
        index_by_id[record_id]
    ]

    batch = ctc_collator([example])
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    # Encoder is frozen.
    with torch.no_grad():
        hidden = base_model.wav2vec2(
            input_values=batch["input_values"],
            attention_mask=batch["attention_mask"]
        ).last_hidden_state

    hidden = hidden.detach().clone().float()

    targets = batch["labels"][
        batch["labels"] >= 0
    ].reshape(-1).contiguous()

    num_frames = hidden.shape[1]
    num_tokens = targets.numel()

    repeated = int(
        (targets[1:] == targets[:-1]).sum().item()
    )

    assert num_frames >= num_tokens + repeated
    assert not (targets == BLANK_ID).any()
    assert not hidden.is_inference()

    reference = normalize_amharic_ctc(
        record_by_id[record_id]["transcription"]
    )

    return {
        "id": record_id,
        "features": hidden,
        "targets": targets,
        "input_lengths": torch.tensor(
            [num_frames], dtype=torch.long
        ),
        "target_lengths": torch.tensor(
            [num_tokens], dtype=torch.long
        ),
        "reference": reference
    }

print("\nCaching 12 recordings...")

train_items = [
    cache_example(rid)
    for rid in train_ids
]

val_items = [
    cache_example(rid)
    for rid in val_ids
]

# No further need for the full encoder.
del base_model
gc.collect()
torch.cuda.empty_cache()

print("Encoder features cached: 12/12")
print(
    "Trainable parameters:",
    sum(p.numel() for p in head.parameters())
)

# ------------------------------------------------------------
# 4. CTC loss and evaluation
# ------------------------------------------------------------

def example_loss(item):
    logits = head(item["features"])

    log_probs = F.log_softmax(
        logits.float(),
        dim=-1
    ).transpose(0, 1)

    return F.ctc_loss(
        log_probs,
        item["targets"],
        item["input_lengths"],
        item["target_lengths"],
        blank=BLANK_ID,
        reduction="mean",
        zero_infinity=False
    )


def evaluate_group(items):
    head.eval()

    references = []
    predictions = []
    losses = []
    nonempty = 0

    with torch.no_grad():

        for item in items:

            logits = head(item["features"])

            predicted_ids = logits.argmax(
                dim=-1
            )

            prediction = processor.batch_decode(
                predicted_ids.cpu()
            )[0]

            prediction = normalize_amharic_ctc(
                prediction
            )

            reference = item["reference"]

            references.append(reference)
            predictions.append(prediction)

            losses.append(
                float(example_loss(item).item())
            )

            if prediction.strip():
                nonempty += 1

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "mean_loss": sum(losses) / len(losses),
        "nonempty": nonempty,
        "count": len(items),
        "references": references,
        "predictions": predictions
    }

optimizer = torch.optim.AdamW(
    head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0
)

# ------------------------------------------------------------
# 5. Resume experiment if previously saved
# ------------------------------------------------------------

history = []
start_step = 0

if CHECKPOINT_PATH.exists():

    state = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True
    )

    assert state["signature"] == selection_signature
    assert state["learning_rate"] == LEARNING_RATE

    head.load_state_dict(state["head"])
    optimizer.load_state_dict(state["optimizer"])

    start_step = int(state["step"])

    if HISTORY_PATH.exists():
        history = json.loads(
            HISTORY_PATH.read_text(encoding="utf-8")
        )

    print("\nResuming from update:", start_step)

else:
    print("\nStarting a new experiment.")

# ------------------------------------------------------------
# 6. Save a resumable checkpoint
# ------------------------------------------------------------

def save_experiment(step):

    state = {
        "step": step,
        "signature": selection_signature,
        "learning_rate": LEARNING_RATE,
        "head": head.state_dict(),
        "optimizer": optimizer.state_dict()
    }

    temp_checkpoint = (
        EXPERIMENT_DIR / "training_latest.pt.partial"
    )

    torch.save(state, temp_checkpoint)
    temp_checkpoint.replace(CHECKPOINT_PATH)

    temp_history = (
        EXPERIMENT_DIR / "training_history.json.partial"
    )

    temp_history.write_text(
        json.dumps(
            history,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    temp_history.replace(HISTORY_PATH)

# ------------------------------------------------------------
# 7. Evaluate before training
# ------------------------------------------------------------

def report_step(step):

    train_result = evaluate_group(train_items)
    val_result = evaluate_group(val_items)

    record = {
        "step": step,
        "train_cer": train_result["cer"],
        "train_wer": train_result["wer"],
        "train_loss": train_result["mean_loss"],
        "train_nonempty": train_result["nonempty"],
        "val_cer": val_result["cer"],
        "val_wer": val_result["wer"],
        "val_loss": val_result["mean_loss"],
        "val_nonempty": val_result["nonempty"]
    }

    print(f"\nUPDATE {step}/{MAX_UPDATES}")
    print("-" * 45)

    print(
        "TRAIN CER:",
        f"{train_result['cer'] * 100:.2f}%",
        "| WER:",
        f"{train_result['wer'] * 100:.2f}%"
    )

    print(
        "HELD-OUT CER:",
        f"{val_result['cer'] * 100:.2f}%",
        "| WER:",
        f"{val_result['wer'] * 100:.2f}%"
    )

    print(
        "Nonempty predictions:",
        f"{train_result['nonempty']}/8 train,",
        f"{val_result['nonempty']}/4 held-out"
    )

    print(
        "Mean train CTC loss:",
        round(train_result["mean_loss"], 4)
    )

    print("\nTRAIN SAMPLE:")
    print(
        train_result["predictions"][0][:150]
        or "[EMPTY]"
    )

    print("HELD-OUT SAMPLE:")
    print(
        val_result["predictions"][0][:150]
        or "[EMPTY]"
    )

    return record

if start_step == 0 and not history:
    history.append(report_step(0))

# ------------------------------------------------------------
# 8. Train on all eight examples in a rotating order
# ------------------------------------------------------------

print("\nBeginning head-only training...")
print("Maximum updates:", MAX_UPDATES)
print("Checkpoint interval:", REPORT_EVERY)

completed_step = start_step

for step in range(start_step + 1, MAX_UPDATES + 1):

    # Deterministic rotation through the eight examples.
    item = train_items[
        (step - 1) % len(train_items)
    ]

    head.train()
    optimizer.zero_grad(set_to_none=True)

    loss = example_loss(item)

    assert torch.isfinite(loss).item()
    assert loss.requires_grad

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        head.parameters(),
        max_norm=5.0,
        error_if_nonfinite=True
    )

    optimizer.step()
    completed_step = step

    if step % REPORT_EVERY == 0:

        result = report_step(step)
        history.append(result)

        save_experiment(step)

# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

final_train = evaluate_group(train_items)
final_val = evaluate_group(val_items)

print("\n" + "=" * 60)
print("FINAL MULTI-RECORDING RESULTS")
print("=" * 60)

print("Completed updates:", completed_step)

print(
    "Train CER:",
    f"{final_train['cer'] * 100:.2f}%"
)

print(
    "Held-out CER:",
    f"{final_val['cer'] * 100:.2f}%"
)

print(
    "Held-out WER:",
    f"{final_val['wer'] * 100:.2f}%"
)

print("\nHELD-OUT EXAMPLE")
print("Reference:")
print(final_val["references"][0])
print("\nPrediction:")
print(final_val["predictions"][0] or "[EMPTY]")

print("\nCheckpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)
print("Original XLS-R and step-500 model unchanged.")


## Original source cell 40

Position 41 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 40 - SHORT-RECORDING CONTROLLED DATASET
#
# 8 training recordings, including our proven sample
# 4 held-out recordings
# All 12 speakers must be distinct
# Recording duration: 3.5 to 8 seconds
#
# No model training or checkpoint changes.
# ============================================================

import json
import random
from pathlib import Path
from IPython.display import Audio, display

SHORT_DIR = DATA_DIR / "short_multirecording_v1"
SHORT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_SELECTION_PATH = SHORT_DIR / "selection.json"

ANCHOR_ID = "6Q0fEziAqyQVIz4J8aws"
SEED = 20261019

TRAIN_COUNT = 8
VAL_COUNT = 4

CONV_KERNELS = (10, 3, 3, 3, 3, 2, 2)
CONV_STRIDES = (5, 2, 2, 2, 2, 2, 2)

def short_output_frames(n_samples):
    length = int(n_samples)
    for kernel, stride in zip(CONV_KERNELS, CONV_STRIDES):
        length = (length - kernel) // stride + 1
    return length

def short_candidate_ok(record):

    duration = float(record["duration_seconds"])
    speaker = record.get("speaker_id")
    transcript = record["transcription"]

    if not speaker:
        return False

    if not (3.5 <= duration <= 8.0):
        return False

    # Avoid digit-to-speech normalization ambiguity
    # in this small controlled experiment.
    if any(character.isdigit() for character in transcript):
        return False

    audio_path = Path(AUDIO_DIR) / record["audio_file"]

    if not audio_path.is_file():
        return False

    labels = tokenizer.encode(
        transcript,
        add_special_tokens=False
    )

    if not (20 <= len(labels) <= 85):
        return False

    if tokenizer.unk_token_id in labels:
        return False

    repeated = sum(
        left == right
        for left, right in zip(labels[:-1], labels[1:])
    )

    required_frames = len(labels) + repeated
    available_frames = short_output_frames(
        round(duration * 16000)
    )

    return available_frames - required_frames >= 30


record_lookup = {
    record["id"]: record
    for record in train_records
}

assert ANCHOR_ID in record_lookup
assert short_candidate_ok(record_lookup[ANCHOR_ID])

# ------------------------------------------------------------
# 1. Reuse selection if already generated
# ------------------------------------------------------------

if SHORT_SELECTION_PATH.exists():

    selection = json.loads(
        SHORT_SELECTION_PATH.read_text(
            encoding="utf-8"
        )
    )

    short_train_ids = selection["train_ids"]
    short_val_ids = selection["val_ids"]

    print("Restored existing short-recording selection.")

else:

    eligible = [
        record
        for record in train_records
        if short_candidate_ok(record)
        and record["id"] != ANCHOR_ID
    ]

    rng = random.Random(SEED)
    rng.shuffle(eligible)

    selected = [record_lookup[ANCHOR_ID]]
    used_speakers = {
        record_lookup[ANCHOR_ID]["speaker_id"]
    }

    for record in eligible:

        if record["speaker_id"] in used_speakers:
            continue

        selected.append(record)
        used_speakers.add(record["speaker_id"])

        if len(selected) == TRAIN_COUNT + VAL_COUNT:
            break

    assert len(selected) == 12, (
        "Not enough suitable recordings."
    )

    short_train_ids = [
        record["id"]
        for record in selected[:TRAIN_COUNT]
    ]

    short_val_ids = [
        record["id"]
        for record in selected[TRAIN_COUNT:]
    ]

    selection = {
        "seed": SEED,
        "anchor_id": ANCHOR_ID,
        "train_ids": short_train_ids,
        "val_ids": short_val_ids,
        "duration_range_seconds": [3.5, 8.0]
    }

    SHORT_SELECTION_PATH.write_text(
        json.dumps(
            selection,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("Eligible short recordings:", len(eligible))
    print("Created new short-recording selection.")

# ------------------------------------------------------------
# 2. Check sample integrity
# ------------------------------------------------------------

assert len(short_train_ids) == TRAIN_COUNT
assert len(short_val_ids) == VAL_COUNT
assert ANCHOR_ID in short_train_ids

all_ids = short_train_ids + short_val_ids

assert len(set(all_ids)) == 12

short_train_records = [
    record_lookup[rid]
    for rid in short_train_ids
]

short_val_records = [
    record_lookup[rid]
    for rid in short_val_ids
]

all_selected = (
    short_train_records + short_val_records
)

assert all(
    short_candidate_ok(record)
    for record in all_selected
)

all_speakers = [
    record["speaker_id"]
    for record in all_selected
]

assert len(set(all_speakers)) == 12

total_train_seconds = sum(
    float(record["duration_seconds"])
    for record in short_train_records
)

total_val_seconds = sum(
    float(record["duration_seconds"])
    for record in short_val_records
)

# ------------------------------------------------------------
# 3. Display selected dataset
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SHORT DATASET SUMMARY")
print("=" * 60)

print("Training recordings:", len(short_train_records))
print("Held-out recordings:", len(short_val_records))
print("Unique speakers:", len(set(all_speakers)))
print("Train duration:", round(total_train_seconds, 2), "seconds")
print("Held-out duration:", round(total_val_seconds, 2), "seconds")
print("Positive control:", ANCHOR_ID)

print("\nTRAINING RECORDINGS")
for index, record in enumerate(short_train_records, 1):
    print(
        f"\n{index}. {record['id']} "
        f"| {record['duration_seconds']:.2f}s"
    )
    print(record["transcription"])

print("\nHELD-OUT RECORDINGS")
for index, record in enumerate(short_val_records, 1):
    print(
        f"\n{index}. {record['id']} "
        f"| {record['duration_seconds']:.2f}s"
    )
    print(record["transcription"])

# ------------------------------------------------------------
# 4. Listen and check audio-to-text quality
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AUDIO QUALITY REVIEW")
print("=" * 60)

review_examples = [
    ("KNOWN POSITIVE CONTROL", short_train_records[0]),
    ("NEW TRAINING EXAMPLE", short_train_records[1]),
    ("NEW TRAINING EXAMPLE", short_train_records[2]),
    ("HELD-OUT EXAMPLE", short_val_records[0])
]

for label, record in review_examples:

    print("\n", label)
    print("ID:", record["id"])
    print("Transcript:", record["transcription"])

    audio_path = Path(AUDIO_DIR) / record["audio_file"]

    display(Audio(filename=str(audio_path)))

print("\nSelection saved:", SHORT_SELECTION_PATH)
print("No existing models or checkpoints were modified.")


## Original source cell 41

Position 42 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 40B - ADAPTIVE SHORT-RECORDING DATASET FIX
# Prefer short recordings, relax duration when necessary.
# No training or checkpoint modifications.
# ============================================================

import json
import random
from pathlib import Path
from IPython.display import Audio, display

ANCHOR_ID = "6Q0fEziAqyQVIz4J8aws"
TRAIN_COUNT = 8
VAL_COUNT = 4

SHORT_DIR = DATA_DIR / "short_multirecording_v1"
SHORT_DIR.mkdir(parents=True, exist_ok=True)
SHORT_SELECTION_PATH = SHORT_DIR / "selection.json"

record_lookup = {r["id"]: r for r in train_records}
assert ANCHOR_ID in record_lookup

kernels = (10, 3, 3, 3, 3, 2, 2)
strides = (5, 2, 2, 2, 2, 2, 2)

def frame_count(samples):
    for kernel, stride in zip(kernels, strides):
        samples = (samples - kernel) // stride + 1
    return samples

def eligible_record(record):
    duration = float(record["duration_seconds"])
    text = record["transcription"]

    if not record.get("speaker_id"):
        return False
    if not 3.5 <= duration <= 18.0:
        return False

    audio_path = Path(AUDIO_DIR) / record["audio_file"]
    if not audio_path.is_file():
        return False

    labels = tokenizer.encode(
        text, add_special_tokens=False
    )

    if not 20 <= len(labels) <= 140:
        return False
    if tokenizer.unk_token_id in labels:
        return False

    repeats = sum(
        a == b for a, b in zip(labels[:-1], labels[1:])
    )

    available = frame_count(round(duration * 16000))
    required = len(labels) + repeats

    return available - required >= 20

# ------------------------------------------------------------
# 1. Find suitable recordings
# ------------------------------------------------------------

candidates = [
    r for r in train_records
    if eligible_record(r)
]

assert eligible_record(record_lookup[ANCHOR_ID]), (
    "The known positive-control recording is unavailable."
)

rng = random.Random(20261019)
rng.shuffle(candidates)

# Prefer shorter recordings.
candidates.sort(
    key=lambda r: float(r["duration_seconds"])
)

print("DATASET AVAILABILITY")
print("-" * 50)

for maximum in [8, 10, 12, 15, 18]:
    speakers = {
        r["speaker_id"]
        for r in candidates
        if float(r["duration_seconds"]) <= maximum
    }
    print(
        f"Up to {maximum:2d}s:",
        f"{len(speakers)} unique speakers"
    )

# ------------------------------------------------------------
# 2. Adaptively select 12 different speakers
# ------------------------------------------------------------

chosen = None
selected_max_duration = None
digits_allowed = None

for allow_digits in [False, True]:

    for maximum in [8, 10, 12, 15, 18]:

        attempt = [record_lookup[ANCHOR_ID]]
        used_speakers = {
            attempt[0]["speaker_id"]
        }

        for record in candidates:

            if record["id"] == ANCHOR_ID:
                continue

            if float(record["duration_seconds"]) > maximum:
                continue

            if not allow_digits and any(
                ch.isdigit()
                for ch in record["transcription"]
            ):
                continue

            speaker = record["speaker_id"]

            if speaker in used_speakers:
                continue

            attempt.append(record)
            used_speakers.add(speaker)

            if len(attempt) == TRAIN_COUNT + VAL_COUNT:
                break

        if len(attempt) == TRAIN_COUNT + VAL_COUNT:
            chosen = attempt
            selected_max_duration = maximum
            digits_allowed = allow_digits
            break

    if chosen is not None:
        break

assert chosen is not None, (
    "Could not find 12 distinct speakers even after "
    "relaxing the recording duration to 18 seconds."
)

# ------------------------------------------------------------
# 3. Save and validate selection
# ------------------------------------------------------------

new_selection = {
    "anchor_id": ANCHOR_ID,
    "train_ids": [r["id"] for r in chosen[:8]],
    "val_ids": [r["id"] for r in chosen[8:]],
    "maximum_duration": selected_max_duration,
    "digits_allowed": digits_allowed
}

if SHORT_SELECTION_PATH.exists():
    selection = json.loads(
        SHORT_SELECTION_PATH.read_text(encoding="utf-8")
    )
    print("\nUsing existing saved selection.")
else:
    selection = new_selection
    SHORT_SELECTION_PATH.write_text(
        json.dumps(
            selection, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("\nNew selection saved.")

short_train_ids = selection["train_ids"]
short_val_ids = selection["val_ids"]

short_train_records = [
    record_lookup[rid] for rid in short_train_ids
]
short_val_records = [
    record_lookup[rid] for rid in short_val_ids
]

all_records = short_train_records + short_val_records

assert len(short_train_records) == 8
assert len(short_val_records) == 4
assert len({r["id"] for r in all_records}) == 12
assert len({r["speaker_id"] for r in all_records}) == 12
assert ANCHOR_ID in short_train_ids
assert all(eligible_record(r) for r in all_records)

print("\n" + "=" * 55)
print("CORRECTED DATASET SUMMARY")
print("=" * 55)

print("Training recordings:", len(short_train_records))
print("Held-out recordings:", len(short_val_records))
print("Unique speakers:", 12)
print(
    "Longest selected recording:",
    round(max(
        float(r["duration_seconds"])
        for r in all_records
    ), 2),
    "seconds"
)
print("Selection duration ceiling:", selected_max_duration)
print("Digits permitted in selection:", digits_allowed)

print("\nTRAINING RECORDINGS")
for i, record in enumerate(short_train_records, 1):
    print(
        f"{i}. {record['id']} | "
        f"{record['duration_seconds']:.2f}s"
    )
    print("   ", record["transcription"])

print("\nHELD-OUT RECORDINGS")
for i, record in enumerate(short_val_records, 1):
    print(
        f"{i}. {record['id']} | "
        f"{record['duration_seconds']:.2f}s"
    )
    print("   ", record["transcription"])

# ------------------------------------------------------------
# 4. Listen to a few examples
# ------------------------------------------------------------

print("\nAUDIO REVIEW")

for label, record in [
    ("KNOWN CONTROL", short_train_records[0]),
    ("NEW TRAINING", short_train_records[1]),
    ("HELD-OUT", short_val_records[0])
]:
    print("\n", label)
    print("Transcript:", record["transcription"])

    display(Audio(
        filename=str(
            Path(AUDIO_DIR) / record["audio_file"]
        )
    ))

print("\nSELECTION VALIDATION PASSED")
print("Saved:", SHORT_SELECTION_PATH)
print("No model checkpoints modified.")


## Original source cell 42

Position 43 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 41 - BALANCED SHORT-RECORDING ASR EXPERIMENT
#
# 8 train / 4 held-out, 12 different speakers
# Duration-balanced split
# Original XLS-R encoder frozen
# Only CTC head trainable
# ============================================================

import json
import copy
import gc
import hashlib
import itertools
import statistics
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MAX_UPDATES = 4000
REPORT_EVERY = 250
HEAD_LR = 1e-3

DEVICE_SHORT = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BLANK_SHORT = tokenizer.pad_token_id
ANCHOR_ID = "6Q0fEziAqyQVIz4J8aws"

SHORT_EXPERIMENT_DIR = (
    DATA_DIR / "short_multirecording_head_only_v2"
)
SHORT_EXPERIMENT_DIR.mkdir(
    parents=True, exist_ok=True
)

SPLIT_PATH = SHORT_EXPERIMENT_DIR / "split.json"
STATE_PATH = SHORT_EXPERIMENT_DIR / "latest.pt"
HISTORY_PATH = SHORT_EXPERIMENT_DIR / "history.json"

original_selection = json.loads(
    (
        DATA_DIR
        / "short_multirecording_v1"
        / "selection.json"
    ).read_text(encoding="utf-8")
)

pool_ids = (
    original_selection["train_ids"]
    + original_selection["val_ids"]
)

assert len(pool_ids) == 12
assert len(set(pool_ids)) == 12
assert ANCHOR_ID in pool_ids

by_id = {r["id"]: r for r in train_records}

def duration(rid):
    return float(by_id[rid]["duration_seconds"])

# ------------------------------------------------------------
# 2. Create a more balanced speaker-disjoint split
# ------------------------------------------------------------

if SPLIT_PATH.exists():

    split = json.loads(
        SPLIT_PATH.read_text(encoding="utf-8")
    )

    short_train_ids = split["train_ids"]
    short_val_ids = split["val_ids"]

    print("Reusing previously saved balanced split.")

else:

    others = [
        rid for rid in pool_ids
        if rid != ANCHOR_ID
    ]

    possible_splits = []

    for val_group in itertools.combinations(others, 4):

        val_group = list(val_group)
        val_set = set(val_group)

        train_group = [
            rid for rid in pool_ids
            if rid not in val_set
        ]

        train_durations = [
            duration(rid) for rid in train_group
        ]
        val_durations = [
            duration(rid) for rid in val_group
        ]

        # Both groups should contain recordings
        # of different lengths.
        if not any(d < 6 for d in val_durations):
            continue
        if not any(d >= 8 for d in val_durations):
            continue
        if not any(d >= 8 for d in train_durations):
            continue

        mean_gap = abs(
            statistics.mean(train_durations)
            - statistics.mean(val_durations)
        )

        spread_gap = abs(
            statistics.pstdev(train_durations)
            - statistics.pstdev(val_durations)
        )

        score = mean_gap + 0.25 * spread_gap

        possible_splits.append(
            (score, tuple(sorted(val_group)),
             train_group, val_group)
        )

    assert possible_splits, (
        "No suitable balanced split found."
    )

    possible_splits.sort(
        key=lambda item: (item[0], item[1])
    )

    _, _, short_train_ids, short_val_ids = (
        possible_splits[0]
    )

    split = {
        "train_ids": short_train_ids,
        "val_ids": short_val_ids,
        "anchor_id": ANCHOR_ID,
        "source_pool_ids": sorted(pool_ids)
    }

    SPLIT_PATH.write_text(
        json.dumps(
            split, ensure_ascii=False, indent=2
        ),
        encoding="utf-8"
    )

assert len(short_train_ids) == 8
assert len(short_val_ids) == 4
assert set(short_train_ids + short_val_ids) == set(pool_ids)
assert ANCHOR_ID in short_train_ids

train_speakers = {
    by_id[rid]["speaker_id"]
    for rid in short_train_ids
}
val_speakers = {
    by_id[rid]["speaker_id"]
    for rid in short_val_ids
}

assert len(train_speakers) == 8
assert len(val_speakers) == 4
assert not train_speakers.intersection(val_speakers)

print("\nBALANCED SPLIT")
print("-" * 55)
print("Training recordings:", len(short_train_ids))
print("Held-out recordings:", len(short_val_ids))
print("Overlapping speakers:", 0)

print(
    "Mean train duration:",
    round(statistics.mean(
        duration(rid) for rid in short_train_ids
    ), 2),
    "seconds"
)

print(
    "Mean held-out duration:",
    round(statistics.mean(
        duration(rid) for rid in short_val_ids
    ), 2),
    "seconds"
)

# ------------------------------------------------------------
# 3. Verify a reproducible experiment signature
# ------------------------------------------------------------

experiment_config = {
    "train_ids": short_train_ids,
    "val_ids": short_val_ids,
    "references": {
        rid: by_id[rid]["transcription"]
        for rid in short_train_ids + short_val_ids
    },
    "baseline": str(BASELINE_DIR),
    "head_lr": HEAD_LR
}

signature = hashlib.sha256(
    json.dumps(
        experiment_config,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

index_by_id = {
    r["id"]: index
    for index, r in enumerate(train_dataset.records)
}

# ------------------------------------------------------------
# 4. Load frozen original XLS-R encoder
# ------------------------------------------------------------

print("\nLoading original XLS-R model...")

short_model = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_SHORT)

short_model.eval()

# Copy the CTC head outside inference mode.
short_head = copy.deepcopy(
    short_model.lm_head
).to(DEVICE_SHORT).float()

assert short_head.weight.requires_grad
assert not short_head.weight.is_inference()

# ------------------------------------------------------------
# 5. Cache encoder features once
# ------------------------------------------------------------

def prepare_short_item(rid):

    example = train_dataset[index_by_id[rid]]
    batch = ctc_collator([example])

    batch = {
        key: value.to(DEVICE_SHORT)
        for key, value in batch.items()
    }

    with torch.no_grad():

        features = short_model.wav2vec2(
            input_values=batch["input_values"],
            attention_mask=batch["attention_mask"]
        ).last_hidden_state

    features = features.detach().clone().float()

    targets = batch["labels"][
        batch["labels"] >= 0
    ].reshape(-1).contiguous()

    repeated = int(
        (targets[1:] == targets[:-1]).sum().item()
    )

    assert features.shape[1] >= (
        targets.numel() + repeated
    )

    assert not features.is_inference()
    assert torch.isfinite(features).all()

    return {
        "id": rid,
        "features": features,
        "targets": targets,
        "input_lengths": torch.tensor(
            [features.shape[1]], dtype=torch.long
        ),
        "target_lengths": torch.tensor(
            [targets.numel()], dtype=torch.long
        ),
        "reference": normalize_amharic_ctc(
            by_id[rid]["transcription"]
        )
    }

print("\nCaching encoder features...")

short_train_items = [
    prepare_short_item(rid)
    for rid in short_train_ids
]

short_val_items = [
    prepare_short_item(rid)
    for rid in short_val_ids
]

del short_model
gc.collect()
torch.cuda.empty_cache()

print("Cached: 8 train + 4 held-out")
print(
    "Trainable head parameters:",
    sum(p.numel() for p in short_head.parameters())
)

# ------------------------------------------------------------
# 6. Training and evaluation functions
# ------------------------------------------------------------

def short_ctc_loss(item):

    logits = short_head(item["features"])

    return F.ctc_loss(
        F.log_softmax(
            logits.float(), dim=-1
        ).transpose(0, 1),
        item["targets"],
        item["input_lengths"],
        item["target_lengths"],
        blank=BLANK_SHORT,
        reduction="mean",
        zero_infinity=False
    )


def short_evaluate(items):

    short_head.eval()

    references = []
    predictions = []
    losses = []

    with torch.no_grad():

        for item in items:

            logits = short_head(item["features"])

            ids = logits.argmax(dim=-1)

            text = processor.batch_decode(
                ids.cpu()
            )[0]

            text = normalize_amharic_ctc(text)

            references.append(item["reference"])
            predictions.append(text)

            losses.append(
                float(short_ctc_loss(item).item())
            )

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "loss": sum(losses) / len(losses),
        "nonempty": sum(
            bool(text.strip()) for text in predictions
        ),
        "predictions": predictions
    }

short_optimizer = torch.optim.AdamW(
    short_head.parameters(),
    lr=HEAD_LR,
    weight_decay=0.0
)

# ------------------------------------------------------------
# 7. Resume if an experiment already exists
# ------------------------------------------------------------

history = []
start_step = 0

if STATE_PATH.exists():

    state = torch.load(
        STATE_PATH,
        map_location="cpu",
        weights_only=True
    )

    assert state["signature"] == signature
    assert state["lr"] == HEAD_LR

    short_head.load_state_dict(state["head"])
    short_optimizer.load_state_dict(state["optimizer"])

    start_step = int(state["step"])

    if HISTORY_PATH.exists():
        history = json.loads(
            HISTORY_PATH.read_text(encoding="utf-8")
        )

    print(f"\nResuming from step {start_step}")

else:
    print("\nStarting new head-only experiment.")

def short_save(step):

    state = {
        "step": step,
        "signature": signature,
        "lr": HEAD_LR,
        "head": short_head.state_dict(),
        "optimizer": short_optimizer.state_dict()
    }

    temp = SHORT_EXPERIMENT_DIR / "latest.pt.partial"

    torch.save(state, temp)
    temp.replace(STATE_PATH)

    temp_hist = (
        SHORT_EXPERIMENT_DIR / "history.json.partial"
    )

    temp_hist.write_text(
        json.dumps(
            history, ensure_ascii=False, indent=2
        ),
        encoding="utf-8"
    )

    temp_hist.replace(HISTORY_PATH)

# ------------------------------------------------------------
# 8. Reporting
# ------------------------------------------------------------

def short_report(step):

    train_result = short_evaluate(short_train_items)
    val_result = short_evaluate(short_val_items)

    result = {
        "step": step,
        "train_cer": train_result["cer"],
        "train_wer": train_result["wer"],
        "train_loss": train_result["loss"],
        "val_cer": val_result["cer"],
        "val_wer": val_result["wer"],
        "val_loss": val_result["loss"],
        "train_nonempty": train_result["nonempty"],
        "val_nonempty": val_result["nonempty"]
    }

    print(f"\nSTEP {step}/{MAX_UPDATES}")
    print("-" * 48)

    print(
        "Train CER:",
        f"{train_result['cer'] * 100:.2f}%"
    )

    print(
        "Held-out CER:",
        f"{val_result['cer'] * 100:.2f}%"
    )

    print(
        "Held-out WER:",
        f"{val_result['wer'] * 100:.2f}%"
    )

    print(
        "Train loss:",
        round(train_result["loss"], 4)
    )

    print(
        "Nonempty:",
        f"{train_result['nonempty']}/8 train,",
        f"{val_result['nonempty']}/4 held-out"
    )

    print("\nKnown positive-control prediction:")
    anchor_index = short_train_ids.index(ANCHOR_ID)
    print(
        train_result["predictions"][anchor_index]
        or "[EMPTY]"
    )

    print("\nHeld-out prediction:")
    print(
        val_result["predictions"][0]
        or "[EMPTY]"
    )

    return result

if start_step == 0 and not history:
    history.append(short_report(0))

# ------------------------------------------------------------
# 9. Train with deterministic rotating examples
# ------------------------------------------------------------

print("\nTRAINING SHORT-RECORDING MODEL")
print("=" * 55)

completed_step = start_step

for step in range(start_step + 1, MAX_UPDATES + 1):

    item = short_train_items[
        (step - 1) % len(short_train_items)
    ]

    short_head.train()
    short_optimizer.zero_grad(set_to_none=True)

    loss = short_ctc_loss(item)

    assert torch.isfinite(loss).item()
    assert loss.requires_grad

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        short_head.parameters(),
        max_norm=5.0,
        error_if_nonfinite=True
    )

    short_optimizer.step()

    completed_step = step

    if step % REPORT_EVERY == 0:

        report = short_report(step)
        history.append(report)

        short_save(step)

# ------------------------------------------------------------
# 10. Final results
# ------------------------------------------------------------

final_train = short_evaluate(short_train_items)
final_val = short_evaluate(short_val_items)

print("\n" + "=" * 60)
print("FINAL SHORT-RECORDING RESULTS")
print("=" * 60)

print("Completed updates:", completed_step)

print(
    "Training CER:",
    f"{final_train['cer'] * 100:.2f}%"
)

print(
    "Held-out CER:",
    f"{final_val['cer'] * 100:.2f}%"
)

print(
    "Held-out WER:",
    f"{final_val['wer'] * 100:.2f}%"
)

print("\nCheckpoint:", STATE_PATH)
print("History:", HISTORY_PATH)

print("\nExisting XLS-R and step-500 checkpoints unchanged.")


## Original source cell 43

Position 44 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 42 - CONTROLLED TWO-STAGE FINE-TUNING TEST
#
# A: Frozen XLS-R encoder + trained CTC head
# B: Train last 2 XLS-R layers + same trained CTC head
#
# Both start from the Cell 41 head at update 4000.
# Same data order, head LR, and number of updates.
# Old checkpoints are never modified.
# ============================================================

import copy
import gc
import json
import hashlib
from pathlib import Path

import torch
import torch.nn.functional as F
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

DEVICE_42 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PILOT_UPDATES = 300
REPORT_EVERY = 100
HEAD_LR = 3e-4
ENCODER_LR = 1e-5
BLANK = tokenizer.pad_token_id
ANCHOR_ID = "6Q0fEziAqyQVIz4J8aws"

WARM_DIR = DATA_DIR / "short_multirecording_head_only_v2"
WARM_STATE = WARM_DIR / "latest.pt"
SPLIT_FILE = WARM_DIR / "split.json"

RUN_DIR = DATA_DIR / "warm_head_encoder_comparison_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

assert WARM_STATE.is_file()
assert SPLIT_FILE.is_file()

warm = torch.load(
    WARM_STATE,
    map_location="cpu",
    weights_only=True
)

assert int(warm["step"]) == 4000

split = json.loads(
    SPLIT_FILE.read_text(encoding="utf-8")
)

train_ids = split["train_ids"]
val_ids = split["val_ids"]

assert len(train_ids) == 8
assert len(val_ids) == 4
assert ANCHOR_ID in train_ids
assert len(set(train_ids + val_ids)) == 12

by_id = {r["id"]: r for r in train_records}
index_by_id = {
    r["id"]: i
    for i, r in enumerate(train_dataset.records)
}

assert len({
    by_id[rid]["speaker_id"]
    for rid in train_ids + val_ids
}) == 12

signature = hashlib.sha256(
    json.dumps({
        "train_ids": train_ids,
        "val_ids": val_ids,
        "warm_signature": warm["signature"],
        "warm_step": warm["step"],
        "head_lr": HEAD_LR,
        "encoder_lr": ENCODER_LR
    }, sort_keys=True).encode()
).hexdigest()

print("CONTROLLED FINE-TUNING EXPERIMENT")
print("=" * 55)
print("Starting head: Cell 41, update 4000")
print("Training recordings:", len(train_ids))
print("Held-out recordings:", len(val_ids))
print("Updates per branch:", PILOT_UPDATES)

# ------------------------------------------------------------
# 1. Load audio once, keeping exactly the same preprocessing
# ------------------------------------------------------------

batches = {}

for rid in train_ids + val_ids:
    example = train_dataset[index_by_id[rid]]
    batch = ctc_collator([example])
    batches[rid] = {
        k: v.to(DEVICE_42)
        for k, v in batch.items()
    }

references = {
    rid: normalize_amharic_ctc(
        by_id[rid]["transcription"]
    )
    for rid in train_ids + val_ids
}

def score_predictions(predict_fn, ids):
    refs = []
    preds = []

    with torch.no_grad():
        for rid in ids:
            token_ids = predict_fn(rid)
            text = processor.batch_decode(
                token_ids.detach().cpu()
            )[0]
            text = normalize_amharic_ctc(text)

            refs.append(references[rid])
            preds.append(text)

    return {
        "cer": float(cer(refs, preds)),
        "wer": float(wer(refs, preds)),
        "predictions": preds
    }

def report(branch, step, predict_fn):
    train_result = score_predictions(
        predict_fn, train_ids
    )
    val_result = score_predictions(
        predict_fn, val_ids
    )

    result = {
        "step": step,
        "train_cer": train_result["cer"],
        "val_cer": val_result["cer"],
        "val_wer": val_result["wer"],
        "anchor_prediction":
            train_result["predictions"][
                train_ids.index(ANCHOR_ID)
            ],
        "val_prediction":
            val_result["predictions"][0]
    }

    print(f"\n{branch} | UPDATE {step}")
    print(
        "Train CER:",
        f"{result['train_cer'] * 100:.2f}%"
    )
    print(
        "Held-out CER:",
        f"{result['val_cer'] * 100:.2f}%"
    )
    print(
        "Held-out WER:",
        f"{result['val_wer'] * 100:.2f}%"
    )
    print(
        "Anchor:",
        result["anchor_prediction"] or "[EMPTY]"
    )
    print(
        "Held-out:",
        result["val_prediction"] or "[EMPTY]"
    )

    return result

def save_state(path, state, history, history_path):
    temp = path.with_suffix(".pt.partial")
    torch.save(state, temp)
    temp.replace(path)

    history_temp = history_path.with_suffix(
        ".json.partial"
    )
    history_temp.write_text(
        json.dumps(
            history,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    history_temp.replace(history_path)

# ------------------------------------------------------------
# 2. Load original encoder and cache frozen features
# ------------------------------------------------------------

print("\nLoading original encoder...")

base = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_42)

base.eval()
base.lm_head.load_state_dict(warm["head"])

features = {}

with torch.no_grad():
    for rid, batch in batches.items():
        hidden = base.wav2vec2(
            input_values=batch["input_values"],
            attention_mask=batch["attention_mask"]
        ).last_hidden_state

        features[rid] = hidden.detach().clone().float()

head_a = copy.deepcopy(base.lm_head)
head_a.requires_grad_(True)

del base
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Branch A: Continue training the head only
# ------------------------------------------------------------

print("\nBRANCH A: FROZEN ENCODER")
print("=" * 55)

optimizer_a = torch.optim.AdamW(
    head_a.parameters(),
    lr=HEAD_LR,
    weight_decay=0.0
)

def predict_a(rid):
    return head_a(features[rid]).argmax(-1)

def loss_a(rid):
    logits = head_a(features[rid])

    target = batches[rid]["labels"]
    target = target[target >= 0].reshape(-1)

    return F.ctc_loss(
        F.log_softmax(logits.float(), -1).transpose(0, 1),
        target,
        torch.tensor(
            [logits.shape[1]], dtype=torch.long
        ),
        torch.tensor(
            [target.numel()], dtype=torch.long
        ),
        blank=BLANK,
        reduction="mean",
        zero_infinity=False
    )

a_path = RUN_DIR / "head_only_latest.pt"
a_history_path = RUN_DIR / "head_only_history.json"

a_step = 0
history_a = []

if a_path.exists():
    saved_a = torch.load(
        a_path, map_location="cpu", weights_only=True
    )
    assert saved_a["signature"] == signature

    head_a.load_state_dict(saved_a["head"])
    optimizer_a.load_state_dict(saved_a["optimizer"])
    a_step = int(saved_a["step"])

    if a_history_path.exists():
        history_a = json.loads(
            a_history_path.read_text(encoding="utf-8")
        )

if not history_a:
    history_a.append(report("A", a_step, predict_a))

for step in range(a_step + 1, PILOT_UPDATES + 1):

    rid = train_ids[(step - 1) % len(train_ids)]

    optimizer_a.zero_grad(set_to_none=True)
    loss = loss_a(rid)

    assert torch.isfinite(loss).item()
    assert loss.requires_grad

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        head_a.parameters(), 5.0
    )

    optimizer_a.step()

    if step % REPORT_EVERY == 0:
        history_a.append(
            report("A", step, predict_a)
        )

        save_state(
            a_path,
            {
                "signature": signature,
                "step": step,
                "head": head_a.state_dict(),
                "optimizer": optimizer_a.state_dict()
            },
            history_a,
            a_history_path
        )

del head_a, optimizer_a, features
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Branch B: Unfreeze last 2 Transformer layers
# ------------------------------------------------------------

print("\nBRANCH B: TRAIN LAST 2 TRANSFORMER LAYERS")
print("=" * 55)

model_b = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_42)

model_b.lm_head.load_state_dict(warm["head"])

# Freeze everything first.
for parameter in model_b.parameters():
    parameter.requires_grad_(False)

# Train only last 2 Transformer layers.
upper_layers = list(
    model_b.wav2vec2.encoder.layers[-2:]
)

for layer in upper_layers:
    for parameter in layer.parameters():
        parameter.requires_grad_(True)

for parameter in model_b.lm_head.parameters():
    parameter.requires_grad_(True)

# Diagnostic training with dropout and SpecAugment off.
# eval() DOES NOT disable gradients; it disables
# stochastic training behaviors such as dropout.
# This keeps comparison with cached features controlled.
model_b.eval()

encoder_parameters = [
    p
    for layer in upper_layers
    for p in layer.parameters()
    if p.requires_grad
]

head_parameters = list(
    model_b.lm_head.parameters()
)

optimizer_b = torch.optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": ENCODER_LR,
            "weight_decay": 0.01
        },
        {
            "params": head_parameters,
            "lr": HEAD_LR,
            "weight_decay": 0.0
        }
    ]
)

use_amp = DEVICE_42.type == "cuda"

scaler_b = torch.amp.GradScaler(
    "cuda", enabled=use_amp
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_b.parameters()
        if p.requires_grad
    )
)

def predict_b(rid):
    return model_b(**batches[rid]).logits.argmax(-1)

b_path = RUN_DIR / "last2_latest.pt"
b_history_path = RUN_DIR / "last2_history.json"

b_step = 0
history_b = []

if b_path.exists():
    saved_b = torch.load(
        b_path, map_location="cpu", weights_only=True
    )
    assert saved_b["signature"] == signature

    model_b.lm_head.load_state_dict(
        saved_b["head"]
    )

    for layer, weights in zip(
        upper_layers, saved_b["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_b.load_state_dict(
        saved_b["optimizer"]
    )

    scaler_b.load_state_dict(saved_b["scaler"])
    b_step = int(saved_b["step"])

    if b_history_path.exists():
        history_b = json.loads(
            b_history_path.read_text(encoding="utf-8")
        )

if not history_b:
    history_b.append(report("B", b_step, predict_b))

for step in range(b_step + 1, PILOT_UPDATES + 1):

    rid = train_ids[(step - 1) % len(train_ids)]

    optimizer_b.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp
    ):
        loss = model_b(**batches[rid]).loss

    assert torch.isfinite(loss).item()
    assert loss.requires_grad

    scaler_b.scale(loss).backward()
    scaler_b.unscale_(optimizer_b)

    torch.nn.utils.clip_grad_norm_(
        encoder_parameters + head_parameters,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    scaler_b.step(optimizer_b)
    scaler_b.update()

    if step % REPORT_EVERY == 0:

        history_b.append(
            report("B", step, predict_b)
        )

        save_state(
            b_path,
            {
                "signature": signature,
                "step": step,
                "head": model_b.lm_head.state_dict(),
                "upper_layers": [
                    layer.state_dict()
                    for layer in upper_layers
                ],
                "optimizer": optimizer_b.state_dict(),
                "scaler": scaler_b.state_dict()
            },
            history_b,
            b_history_path
        )

# ------------------------------------------------------------
# 5. Side-by-side results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CONTROLLED COMPARISON RESULTS")
print("=" * 60)

print(
    f"{'Update':<10}"
    f"{'A Train':<13}"
    f"{'B Train':<13}"
    f"{'A Held-out':<15}"
    f"{'B Held-out':<15}"
)

a_by_step = {
    row["step"]: row for row in history_a
}
b_by_step = {
    row["step"]: row for row in history_b
}

for step in sorted(
    set(a_by_step) & set(b_by_step)
):
    a = a_by_step[step]
    b = b_by_step[step]

    print(
        f"{step:<10}"
        f"{a['train_cer']*100:<13.2f}"
        f"{b['train_cer']*100:<13.2f}"
        f"{a['val_cer']*100:<15.2f}"
        f"{b['val_cer']*100:<15.2f}"
    )

print("\nBranch A: Frozen pretrained encoder")
print("Branch B: Train last 2 Transformer layers")
print("Previous baseline and step-500 checkpoints unchanged.")
print("Experiment directory:", RUN_DIR)

## Original source cell 44

Position 45 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 42B - REPAIR BRANCH B: FULL FP32 TRAINING
#
# Fix: Disable mixed precision and gradient scaling.
# Reuse the loaded Branch B model and optimizer.
# Branch A and all previous checkpoints remain unchanged.
# ============================================================

import json
import torch
from pathlib import Path

# ------------------------------------------------------------
# 1. Check dependencies from Cell 42
# ------------------------------------------------------------

required = [
    "model_b",
    "optimizer_b",
    "batches",
    "train_ids",
    "val_ids",
    "upper_layers",
    "encoder_parameters",
    "head_parameters",
    "predict_b",
    "report",
    "RUN_DIR",
    "signature",
]

missing = [
    name for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell 42 variables: " + ", ".join(missing)
)

FP32_UPDATES = 300
FP32_REPORT_EVERY = 100

FP32_STATE = RUN_DIR / "last2_fp32_latest.pt"
FP32_HISTORY = RUN_DIR / "last2_fp32_history.json"

# ------------------------------------------------------------
# 2. Prepare the model for full-precision training
# ------------------------------------------------------------

model_b = model_b.float()
model_b.eval()

# Clear invalid gradients left by the failed FP16 attempt.
optimizer_b.zero_grad(set_to_none=True)

trainable_params = [
    p for p in model_b.parameters()
    if p.requires_grad
]

assert len(trainable_params) > 0
assert all(p.dtype == torch.float32 for p in trainable_params)

print("FP32 TRAINING REPAIR")
print("=" * 55)
print("Model parameter dtype: float32")
print("Mixed precision: DISABLED")
print("Gradient scaler: DISABLED")
print("Trainable parameters:", sum(
    p.numel() for p in trainable_params
))

# ------------------------------------------------------------
# 3. Restore previous FP32 progress if available
# ------------------------------------------------------------

fp32_start = 0
fp32_history = []

if FP32_STATE.exists():

    saved = torch.load(
        FP32_STATE,
        map_location="cpu",
        weights_only=True
    )

    assert saved["signature"] == signature
    assert saved["precision"] == "fp32"

    model_b.lm_head.load_state_dict(saved["head"])

    for layer, weights in zip(
        upper_layers,
        saved["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_b.load_state_dict(saved["optimizer"])

    fp32_start = int(saved["step"])

    if FP32_HISTORY.exists():
        fp32_history = json.loads(
            FP32_HISTORY.read_text(encoding="utf-8")
        )

    print("Resuming FP32 experiment:", fp32_start)

else:
    # The FP16 attempt failed before optimizer.step().
    # Therefore the model should still have the
    # original warm-head initialization.
    assert "warm" in globals()

    for name, parameter in model_b.lm_head.state_dict().items():
        expected = warm["head"][name].to(
            parameter.device
        )
        assert torch.equal(parameter, expected), (
            f"Unexpected modification to head: {name}"
        )

    print("Original warm-head weights: VERIFIED")

# ------------------------------------------------------------
# 4. Check one FP32 backward pass WITHOUT updating weights
# ------------------------------------------------------------

first_id = train_ids[
    fp32_start % len(train_ids)
]

print("\nTESTING FP32 GRADIENTS")
print("-" * 55)
print("Diagnostic recording:", first_id)

optimizer_b.zero_grad(set_to_none=True)

# No autocast, no GradScaler.
with torch.autocast(
    device_type="cuda",
    enabled=False
):
    test_loss = model_b(**batches[first_id]).loss

print("Loss:", float(test_loss.item()))
print("Loss dtype:", test_loss.dtype)
print("Requires grad:", test_loss.requires_grad)

assert torch.isfinite(test_loss).item()
assert test_loss.requires_grad

test_loss.backward()

bad_gradients = []

for name, parameter in model_b.named_parameters():

    if not parameter.requires_grad:
        continue

    grad = parameter.grad

    if grad is None:
        continue

    if not torch.isfinite(grad).all().item():

        bad_count = int(
            (~torch.isfinite(grad)).sum().item()
        )

        bad_gradients.append((name, bad_count))

print("Parameters with invalid gradients:",
      len(bad_gradients))

if bad_gradients:
    for name, count in bad_gradients[:10]:
        print("INVALID:", name, "count:", count)

    optimizer_b.zero_grad(set_to_none=True)

    raise RuntimeError(
        "Non-finite gradients persist in FP32. "
        "Do not train yet; inspect the output."
    )

test_grad_norm = torch.nn.utils.clip_grad_norm_(
    trainable_params,
    max_norm=1.0,
    error_if_nonfinite=True
)

print("Gradient norm before clipping:",
      float(test_grad_norm))

optimizer_b.zero_grad(set_to_none=True)

print("\nFP32 GRADIENT CHECK: PASSED")
print("No optimizer update was performed in this test.")

# ------------------------------------------------------------
# 5. Define independent FP32 checkpoint saving
# ------------------------------------------------------------

def save_fp32(step):

    state = {
        "signature": signature,
        "precision": "fp32",
        "step": step,
        "head": model_b.lm_head.state_dict(),
        "upper_layers": [
            layer.state_dict()
            for layer in upper_layers
        ],
        "optimizer": optimizer_b.state_dict(),
    }

    temporary = RUN_DIR / "last2_fp32_latest.pt.partial"

    torch.save(state, temporary)
    temporary.replace(FP32_STATE)

    history_temp = (
        RUN_DIR / "last2_fp32_history.json.partial"
    )

    history_temp.write_text(
        json.dumps(
            fp32_history,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    history_temp.replace(FP32_HISTORY)

# ------------------------------------------------------------
# 6. Evaluate the starting point
# ------------------------------------------------------------

if not fp32_history:
    fp32_history.append(
        report("B FP32", fp32_start, predict_b)
    )

# ------------------------------------------------------------
# 7. Train last two Transformer layers in FP32
# ------------------------------------------------------------

print("\nSTARTING BRANCH B FP32 TRAINING")
print("=" * 55)

completed = fp32_start

for step in range(
    fp32_start + 1,
    FP32_UPDATES + 1
):

    rid = train_ids[
        (step - 1) % len(train_ids)
    ]

    optimizer_b.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        enabled=False
    ):
        loss = model_b(**batches[rid]).loss

    assert torch.isfinite(loss).item(), (
        f"Non-finite loss at update {step}"
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_params,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer_b.step()

    completed = step

    if step % FP32_REPORT_EVERY == 0:

        result = report(
            "B FP32",
            step,
            predict_b
        )

        result["gradient_norm"] = float(
            grad_norm
        )

        result["training_loss"] = float(
            loss.item()
        )

        fp32_history.append(result)

        save_fp32(step)

# ------------------------------------------------------------
# 8. Compare with the saved Branch A results
# ------------------------------------------------------------

a_history_file = RUN_DIR / "head_only_history.json"

assert a_history_file.exists()

history_a_saved = json.loads(
    a_history_file.read_text(encoding="utf-8")
)

a_by_step = {
    row["step"]: row
    for row in history_a_saved
}

b_by_step = {
    row["step"]: row
    for row in fp32_history
}

print("\n" + "=" * 65)
print("CONTROLLED COMPARISON: BRANCH A VS BRANCH B")
print("=" * 65)

print(
    f"{'Step':<8}"
    f"{'A Train':<13}"
    f"{'B Train':<13}"
    f"{'A Held-out':<15}"
    f"{'B Held-out':<15}"
)

for step in sorted(
    set(a_by_step) & set(b_by_step)
):

    a = a_by_step[step]
    b = b_by_step[step]

    print(
        f"{step:<8}"
        f"{a['train_cer']*100:<13.2f}"
        f"{b['train_cer']*100:<13.2f}"
        f"{a['val_cer']*100:<15.2f}"
        f"{b['val_cer']*100:<15.2f}"
    )

print("\nFP32 updates completed:", completed)
print("FP32 checkpoint:", FP32_STATE)
print("FP32 history:", FP32_HISTORY)
print("\nPrevious checkpoints remain unchanged.")


## Original source cell 45

Position 46 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 42C - CLEAN BRANCH B RESET AND FP32 GRADIENT TEST
#
# Rebuild from original XLS-R + saved Cell 41 head.
# Do not reuse potentially inconsistent model weights.
# No training updates; no checkpoints overwritten.
# ============================================================

import gc
import torch
from pathlib import Path
from transformers import Wav2Vec2ForCTC

DEVICE_42 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WARM_STATE_PATH = (
    DATA_DIR
    / "short_multirecording_head_only_v2"
    / "latest.pt"
)

assert WARM_STATE_PATH.is_file()
assert Path(BASELINE_DIR).is_dir()

# ==========================================
# 1. LOAD VERIFIED HEAD WEIGHTS FROM DISK
# ==========================================

warm_verified = torch.load(
    WARM_STATE_PATH,
    map_location="cpu",
    weights_only=True
)

assert warm_verified["step"] == 4000

expected_head = warm_verified["head"]

print("CHECKING PREVIOUS WEIGHT MISMATCH")
print("-" * 55)

if "model_b" in globals():

    actual = (
        model_b.lm_head.weight
        .detach()
        .cpu()
    )

    expected = expected_head["weight"]

    print(
        "Existing head matches saved head:",
        torch.equal(actual, expected)
    )

    print(
        "Current head finite:",
        bool(torch.isfinite(actual).all())
    )

    print(
        "Saved head finite:",
        bool(torch.isfinite(expected).all())
    )

    if torch.isfinite(actual).all() and torch.isfinite(expected).all():
        print(
            "Maximum absolute weight difference:",
            float((actual - expected).abs().max())
        )

# ==========================================
# 2. RELEASE OLD FAILED BRANCH B STATE
# ==========================================

for name in [
    "optimizer_b",
    "scaler_b",
    "model_b",
    "upper_layers",
    "encoder_parameters",
    "head_parameters",
    "trainable_params"
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==========================================
# 3. REBUILD MODEL FROM ORIGINAL BASELINE
# ==========================================

print("\nREBUILDING BRANCH B")
print("-" * 55)

model_b = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_42).float()

# Restore exactly the same head that Branch A used.
model_b.lm_head.load_state_dict(
    expected_head,
    strict=True
)

# Verify that restoration really worked.
for name, parameter in model_b.lm_head.state_dict().items():

    expected = expected_head[name].to(
        parameter.device
    )

    assert torch.equal(parameter, expected), (
        f"Head restoration failed: {name}"
    )

print("Saved Cell 41 head restored: VERIFIED")

# ==========================================
# 4. CONFIGURE TRAINABLE PARAMETERS
# ==========================================

# Freeze the entire model.
model_b.requires_grad_(False)

# Unfreeze last two Transformer layers.
upper_layers = list(
    model_b.wav2vec2.encoder.layers[-2:]
)

for layer in upper_layers:
    layer.requires_grad_(True)

# Unfreeze the CTC head.
model_b.lm_head.requires_grad_(True)

encoder_parameters = [
    parameter
    for layer in upper_layers
    for parameter in layer.parameters()
    if parameter.requires_grad
]

head_parameters = list(
    model_b.lm_head.parameters()
)

trainable_params = (
    encoder_parameters + head_parameters
)

# Disable dropout and SpecAugment for this
# controlled comparison.
# eval() does not disable gradient calculations.
model_b.eval()

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)

print(
    "Trainable Transformer layers:",
    len(upper_layers)
)

# ==========================================
# 5. VERIFY STARTING PREDICTIONS
# ==========================================

assert "batches" in globals()
assert "train_ids" in globals()
assert "val_ids" in globals()
assert "report" in globals()
assert "predict_b" in globals()

print("\nCHECKING STARTING MODEL")

starting_result = report(
    "B CLEAN FP32",
    0,
    predict_b
)

# These should match Cell 41's final metrics.
assert abs(
    starting_result["train_cer"] - 0.4989
) < 0.002

assert abs(
    starting_result["val_cer"] - 0.8761
) < 0.002

print("Starting prediction comparison: PASSED")

# ==========================================
# 6. RUN ONE FP32 BACKWARD PASS
# ==========================================

print("\nFP32 GRADIENT DIAGNOSTIC")
print("-" * 55)

test_id = train_ids[0]

model_b.zero_grad(set_to_none=True)

with torch.autocast(
    device_type=DEVICE_42.type,
    enabled=False
):
    output = model_b(**batches[test_id])
    test_loss = output.loss

print("Test recording:", test_id)
print("CTC loss:", float(test_loss.item()))
print("Loss dtype:", test_loss.dtype)
print("Loss requires gradient:", test_loss.requires_grad)

assert torch.isfinite(test_loss).item()
assert test_loss.requires_grad

test_loss.backward()

# ==========================================
# 7. INSPECT EACH PARAMETER'S GRADIENTS
# ==========================================

invalid_gradients = []
missing_gradients = []
valid_gradient_count = 0

for name, parameter in model_b.named_parameters():

    if not parameter.requires_grad:
        continue

    if parameter.grad is None:
        missing_gradients.append(name)
        continue

    if not torch.isfinite(parameter.grad).all():

        invalid_count = int(
            (~torch.isfinite(parameter.grad))
            .sum().item()
        )

        invalid_gradients.append({
            "parameter": name,
            "invalid_values": invalid_count
        })

    else:
        valid_gradient_count += 1

print(
    "\nParameters with valid gradients:",
    valid_gradient_count
)

print(
    "Parameters with missing gradients:",
    len(missing_gradients)
)

print(
    "Parameters with invalid gradients:",
    len(invalid_gradients)
)

if missing_gradients:
    print("\nFirst missing gradients:")
    for name in missing_gradients[:5]:
        print(" ", name)

if invalid_gradients:
    print("\nFirst invalid gradients:")
    for item in invalid_gradients[:10]:
        print(
            item["parameter"],
            "invalid values:",
            item["invalid_values"]
        )

    model_b.zero_grad(set_to_none=True)

    raise RuntimeError(
        "FP32 gradient diagnostic failed. "
        "Do not start training."
    )

# ==========================================
# 8. VERIFY GRADIENT CLIPPING
# ==========================================

gradient_norm = torch.nn.utils.clip_grad_norm_(
    trainable_params,
    max_norm=1.0,
    error_if_nonfinite=True
)

print(
    "\nOriginal gradient norm:",
    float(gradient_norm)
)

model_b.zero_grad(set_to_none=True)

print("\n" + "=" * 55)
print("CLEAN FP32 MODEL: GRADIENT CHECK PASSED")
print("=" * 55)

print("Original XLS-R weights loaded: YES")
print("Cell 41 head restored exactly: YES")
print("Last 2 Transformer layers trainable: YES")
print("All computed gradients finite: YES")
print("Optimizer updates performed: 0")
print("Previous checkpoints modified: NO")


## Original source cell 46

Position 47 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 42D - BRANCH B FULL FP32 TRAINING
#
# Starting point:
#   Original XLS-R encoder
#   Cell 41 head at update 4000
#
# Trainable:
#   Last 2 Transformer layers
#   CTC prediction head
#
# Compare against completed Branch A.
# ============================================================

import json
import torch
from pathlib import Path

B_MAX_UPDATES = 300
B_REPORT_EVERY = 100

B_HEAD_LR = 3e-4
B_ENCODER_LR = 1e-5

B_STATE_PATH = RUN_DIR / "last2_fp32_latest.pt"
B_HISTORY_PATH = RUN_DIR / "last2_fp32_history.json"

A_HISTORY_PATH = RUN_DIR / "head_only_history.json"

# ------------------------------------------------------------
# 1. Verify the successful Cell 42C state
# ------------------------------------------------------------

assert "model_b" in globals()
assert "warm_verified" in globals()
assert "train_ids" in globals()
assert "batches" in globals()
assert "report" in globals()
assert "predict_b" in globals()

assert A_HISTORY_PATH.is_file()
assert model_b.training is False

model_b.float()
model_b.eval()

b_trainable = [
    p for p in model_b.parameters()
    if p.requires_grad
]

assert sum(p.numel() for p in b_trainable) == 25521473

assert all(
    p.dtype == torch.float32
    for p in b_trainable
)

# ------------------------------------------------------------
# 2. Create a clean optimizer
# ------------------------------------------------------------

optimizer_b = torch.optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": B_ENCODER_LR,
            "weight_decay": 0.01
        },
        {
            "params": head_parameters,
            "lr": B_HEAD_LR,
            "weight_decay": 0.0
        }
    ]
)

optimizer_b.zero_grad(set_to_none=True)

# ------------------------------------------------------------
# 3. Resume if an FP32 checkpoint already exists
# ------------------------------------------------------------

b_start_step = 0
b_fp32_history = []

if B_STATE_PATH.exists():

    saved = torch.load(
        B_STATE_PATH,
        map_location="cpu",
        weights_only=True
    )

    assert saved["signature"] == signature
    assert saved["precision"] == "fp32"

    model_b.lm_head.load_state_dict(
        saved["head"]
    )

    assert len(saved["upper_layers"]) == 2

    for layer, weights in zip(
        upper_layers,
        saved["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_b.load_state_dict(
        saved["optimizer"]
    )

    b_start_step = int(saved["step"])

    assert 0 <= b_start_step <= B_MAX_UPDATES

    if B_HISTORY_PATH.exists():
        b_fp32_history = json.loads(
            B_HISTORY_PATH.read_text(encoding="utf-8")
        )

    print("Resuming FP32 training at:", b_start_step)

else:

    # Verify exact saved head before starting.
    for name, parameter in model_b.lm_head.state_dict().items():
        expected = warm_verified["head"][name].to(
            parameter.device
        )

        assert torch.equal(parameter, expected), (
            f"Starting head mismatch: {name}"
        )

    print("Original Cell 41 head: VERIFIED")
    print("Fresh optimizer: INITIALIZED")

# ------------------------------------------------------------
# 4. Check initial or resumed model
# ------------------------------------------------------------

print("\nBRANCH B FP32 CONFIGURATION")
print("=" * 55)

print("Starting step:", b_start_step)
print("Maximum updates:", B_MAX_UPDATES)
print("Precision: FP32")
print("Head learning rate:", B_HEAD_LR)
print("Encoder learning rate:", B_ENCODER_LR)
print("Trainable parameters:",
      sum(p.numel() for p in b_trainable))

if not b_fp32_history:

    start_metrics = report(
        "B FP32",
        b_start_step,
        predict_b
    )

    b_fp32_history.append(start_metrics)

    if b_start_step == 0:
        assert abs(
            start_metrics["train_cer"] - 0.4989
        ) < 0.002

        assert abs(
            start_metrics["val_cer"] - 0.8761
        ) < 0.002

# ------------------------------------------------------------
# 5. Save progress without touching old models
# ------------------------------------------------------------

def save_branch_b_fp32(step):

    state = {
        "signature": signature,
        "precision": "fp32",
        "step": step,
        "head": model_b.lm_head.state_dict(),
        "upper_layers": [
            layer.state_dict()
            for layer in upper_layers
        ],
        "optimizer": optimizer_b.state_dict()
    }

    temporary = RUN_DIR / "last2_fp32_latest.pt.partial"

    torch.save(state, temporary)
    temporary.replace(B_STATE_PATH)

    temporary_history = (
        RUN_DIR / "last2_fp32_history.json.partial"
    )

    temporary_history.write_text(
        json.dumps(
            b_fp32_history,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    temporary_history.replace(B_HISTORY_PATH)

# ------------------------------------------------------------
# 6. Run 300 full-precision optimizer updates
# ------------------------------------------------------------

print("\nTRAINING BRANCH B")
print("=" * 55)

b_completed = b_start_step

for step in range(
    b_start_step + 1,
    B_MAX_UPDATES + 1
):

    # Identical recording order to Branch A.
    rid = train_ids[
        (step - 1) % len(train_ids)
    ]

    optimizer_b.zero_grad(set_to_none=True)

    # Keep dropout and SpecAugment disabled.
    # eval() does not stop backpropagation.
    model_b.eval()

    # Explicitly disable mixed precision.
    with torch.autocast(
        device_type=DEVICE_42.type,
        enabled=False
    ):
        loss = model_b(**batches[rid]).loss

    assert torch.isfinite(loss).item(), (
        f"Invalid FP32 loss at update {step}"
    )

    assert loss.requires_grad

    loss.backward()

    # Stop immediately if a gradient is NaN or Inf.
    grad_norm = torch.nn.utils.clip_grad_norm_(
        b_trainable,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer_b.step()

    b_completed = step

    if step % B_REPORT_EVERY == 0:

        result = report(
            "B FP32",
            step,
            predict_b
        )

        result["training_loss"] = float(
            loss.item()
        )

        result["gradient_norm"] = float(
            grad_norm
        )

        b_fp32_history.append(result)

        save_branch_b_fp32(step)

        print(
            "Pre-clipping gradient norm:",
            round(float(grad_norm), 4)
        )

        print(
            "Training loss:",
            round(float(loss.item()), 4)
        )

# ------------------------------------------------------------
# 7. Compare saved Branch A with Branch B
# ------------------------------------------------------------

history_a = json.loads(
    A_HISTORY_PATH.read_text(encoding="utf-8")
)

a_by_step = {
    int(row["step"]): row
    for row in history_a
}

b_by_step = {
    int(row["step"]): row
    for row in b_fp32_history
}

print("\n" + "=" * 70)
print("BRANCH A VS BRANCH B")
print("=" * 70)

print(
    f"{'Step':<9}"
    f"{'A Train':<13}"
    f"{'B Train':<13}"
    f"{'A Held-out':<16}"
    f"{'B Held-out':<16}"
)

for step in sorted(
    set(a_by_step) & set(b_by_step)
):

    a = a_by_step[step]
    b = b_by_step[step]

    print(
        f"{step:<9}"
        f"{a['train_cer']*100:<13.2f}"
        f"{b['train_cer']*100:<13.2f}"
        f"{a['val_cer']*100:<16.2f}"
        f"{b['val_cer']*100:<16.2f}"
    )

print("\nFP32 optimizer updates completed:", b_completed)
print("FP32 checkpoint:", B_STATE_PATH)
print("FP32 history:", B_HISTORY_PATH)
print("Previous checkpoints unchanged.")
